# GPT-2 from Scratch — Walkthrough

이 노트북은 Bigram → MLP → GPT-style target → masked self-attention → Tiny GPT로 이어지는 학습 내용을 저장소 코드와 연결합니다.

이 구현은 **문자 단위 Tiny Shakespeare 교육용 GPT-2 스타일 모델**입니다. 공식 GPT-2 checkpoint를 재현하는 것이 아니라 핵심 decoder-only Transformer 구조를 직접 구현합니다.

## 1. 설치

저장소 루트에서 실행하세요.

In [ ]:
%pip install -e .

## 2. 데이터와 shifted target

입력과 target은 한 token만큼 이동합니다.

```text
x = [t1, t2, ..., tT]
y = [t2, t3, ..., t(T+1)]
```

In [ ]:
import torch
from gpt2_scratch.data import (
    CharTokenizer,
    NextTokenDataset,
    download_tiny_shakespeare,
)

path = download_tiny_shakespeare("data/input.txt")
text = path.read_text(encoding="utf-8")
tokenizer = CharTokenizer.from_text(text)
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

dataset = NextTokenDataset(data, block_size=16)
x, y = dataset[0]
print("x:", repr(tokenizer.decode(x.tolist())))
print("y:", repr(tokenizer.decode(y.tolist())))
print("vocab size:", tokenizer.vocab_size)

## 3. GPT-2 스타일 모델

모델은 다음 요소로 구성됩니다.

- token embedding + learned position embedding
- causal multi-head self-attention
- pre-LayerNorm residual block
- GELU feed-forward network
- final LayerNorm + language-model head
- token embedding / output weight tying

In [ ]:
from gpt2_scratch.config import GPTConfig
from gpt2_scratch.model import GPT

config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=64,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1,
)
model = GPT(config)
print(f"parameters: {model.num_parameters():,}")

## 4. 출력 shape와 loss

모든 sequence 위치에서 다음 token을 예측하므로 logits는 `(B, T, V)`입니다.

In [ ]:
xb = torch.stack([dataset[i][0] for i in range(4)])
yb = torch.stack([dataset[i][1] for i in range(4)])

# 현재 dataset 예시는 T=16이므로 model block_size=64 안에 들어갑니다.
logits, loss = model(xb, yb)
print("input:", xb.shape)
print("target:", yb.shape)
print("logits:", logits.shape)
print("loss:", float(loss))

## 5. Causal mask 확인

미래 token을 바꾸어도 그보다 앞선 위치의 logits는 변하지 않아야 합니다.

In [ ]:
model.eval()
a = xb[:1].clone()
b = a.clone()
b[:, 8:] = torch.randint(0, tokenizer.vocab_size, b[:, 8:].shape)

with torch.no_grad():
    logits_a, _ = model(a)
    logits_b, _ = model(b)

print("positions 0..7 unchanged:", torch.allclose(
    logits_a[:, :8], logits_b[:, :8], atol=1e-6
))

## 6. 학습

전체 학습은 CLI 사용을 권장합니다.

```bash
python -m gpt2_scratch.train --config configs/tiny_shakespeare.json
```

Validation loss가 개선될 때 checkpoint가 저장됩니다.

## 7. 생성

학습된 checkpoint가 있다면 다음 명령으로 생성합니다.

```bash
python -m gpt2_scratch.generate \
  --checkpoint checkpoints/tiny_shakespeare.pt \
  --prompt "ROMEO:" \
  --temperature 0.8 \
  --top-k 40
```

생성에서는 실제 prompt만 입력하며 가짜 왼쪽 zero-padding을 사용하지 않습니다. 문맥이 길어지면 최근 `block_size` token만 사용합니다.

## 8. Notebook 1~6 연결

| 단계 | 모델이 보는 문맥 | 핵심 변화 |
|---|---|---|
| Bigram | 현재 문자 1개 | 문자 전이표 |
| MLP names | 최근 3문자 | embedding + MLP |
| MLP Shakespeare | 고정 길이 문맥 | sliding-window 문서 dataset |
| Minimal sequence LM | 자기 위치 token | sequence target/output |
| Single-head attention | 현재와 과거 전체 | causal attention |
| Tiny GPT | 현재와 과거 전체 | multi-head, FFN, residual, LN, stacking |

핵심 변화는 `P(next | current)`에서 `P(next | all previous context)`로 넘어가는 것입니다.